**Disclaimer:** This notebook was developed for students of the "Music Technology Lab" class at Universitat Pompeu Fabra, to help them use the SCAPES model from Google Colab. If you are not in this class or want to use SCAPES locally, prefer the other notebook.

# **Full Pipeline**

This notebook combines the three separate tasks into a single, unified workflow. The main goal is to avoid the need to move files between different notebooks and sessions, which can be time-consuming and inconvenient—especially given the limited time available during class.

By keeping everything in one place, the workflow becomes more streamlined and easier to follow, reducing setup overhead and potential errors caused by file transfers.

That said, when working on these tasks independently, it is recommended to still think of them as separate steps and execute them one by one in order. While combining them here improves convenience for the classroom setting, the structure is intentionally more condensed and may feel less organized for long-term or production use.

# **Preamble**

## Check if your runtime has a GPU

In [ ]:
!nvidia-smi

## Install the model

Note: You might be prompted to restart the kernel after installing it. In such a case, make sure to run this cell again.

In [ ]:
# Clone the specific branch
!git clone --branch colab_version https://github.com/cordutie/SCAPES.git

# Move into repo root
%cd SCAPES

# Install dependencies (if you have requirements.txt)
!pip install -r requirements.txt

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ SCAPES setup complete. You can now do: from SCAPES import ...")

## Mount your Google Drive (optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# **Task 1. Dataset Maker**


Follow this guide to make your own custom dataset using default and previously tested parameters for all steps. 

**Discalimer:** It is recomended to have a copy of your data to start over since some of these steps are not revertible :O (even though no destructive editing to your data is applied).

## **1. Dataset Structure and Configuration**

### **Structure**

Before running any code you must make sure your data is organized as below. 

```
main/
├── raw/
│   ├── audio1.wav
│   ├── audio2.wav
│   └── ...
└── config/
    └── dataprep.json
```

### **Configuration**

You can create the configuration file `config/dataprep.json` by simple creating a `.txt` file, filling it with the text below, and then change its extension to `.json`.

```json
{
    "atoms_frames": 48,
    "atoms_hop_frames": 15,
    "crossfade_frames": 3
}
```

**Note:** You can eliminate train_split and val_split entries if you don't have premade splits.

## **2. Data Preparation**

From here, the data will be prepared in many steps. Make sure you write the path to your dataset in the following cell and then run the whole thing one by one. Some cells may take some time :)

### **2.1. Fix your dataset path**

In [ ]:
# Fill your dataset path here
path_to_dataset = "path/to/your/dataset"  # Change this to your dataset path <---------------------------

### **2.2. Precompute audio representation**

In [ ]:
from SCAPES.data.dataprep import atoms_maker

atoms_maker(path_to_dataset)

### **2.3. Initialize the dataset and split it**

By default, we will use 10 % of all data for training.

In [ ]:
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
import torch

# Pick your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize 48kHz processor
processor_48k_streamable = EncodecProcessor(sr=48000, streamable=True, device=device)

# 1. Default configuration
segment_length, context_length, hop_size = 5, 5, 1

# 2. Setup the main dataset with the new opt-in requested_keys
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=["latent_past", "latent_present", "latent_context_win", 
                    "scale_past", "scale_present", "scale_context_win", 
                    "index"],
    device=device)

dataset.make_split(val_split=0.1)

### **2.4. Precompute semantic annotations using CLAP**

In [ ]:
from SCAPES.data.dataprep import precompute_annotations
from SCAPES.models.factorization import load_global_encoder
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper

method = "clap"
model = CLAPWrapper(version="2023", use_cuda=True)
precompute_annotations(
    dataset=dataset, 
    annotation_type=method, # "clap" or "custom"
    time_part="context_win",  # "past" or "context_win"
    model=model, 
    batch_size=128,
    device="cuda"
)

## **3. Visualize your annotations (optional)**

In [ ]:
from SCAPES.data.visualization import LatentSpaceExplorer
from SCAPES.data.dataset import AtomSequenceDataset

segment_length, context_length, hop_size = 5, 5, 1

# 2. Setup the main dataset with the new opt-in requested_keys
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=["latent_past", "latent_present", "latent_context_win", 
                    "scale_past", "scale_present", "scale_context_win", 
                    "clap_context_win", "index"],
    device="cpu")

# Initialize the visualizer (it will automatically detect the keys)
explorer = LatentSpaceExplorer(dataset=dataset, max_samples_per_file=300)

# Plot using PCA (Fast, shows global structure)
explorer.plot(method="pca")

# # Plot using t-SNE (Slower, but shows local clusters and "islands" beautifully)
explorer.plot(method="tsne")

## **4. Save your dataset**

You will need your preprocessed data for training. Run the following cells to zip your preprocessed data and then **make sure you download it**. 

Note: You can technically just leave it in your Google Drive folder and then train loading the data from there, however that is much slower.

In [ ]:
# Zip the full dataset directory (recursively) using Linux zip
zip_path = f"{path_to_dataset.rstrip('/')}_preprocessed.zip"
!zip -r "{zip_path}" "{path_to_dataset}"

print(f"✅ Created: {zip_path}")

## **5. Restart the kernel (optional)**

It is maybe a good idea to restart the kernel to clean your variables :) If you do so, run the following cell so SCAPES can be loaded properly.

In [ ]:
# Move into repo root
%cd SCAPES

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ SCAPES setup complete. You can now do: from SCAPES import ...")

# **Task 2. Trainer**

## **1. Dataset initialization**

As in the previous notebook. First set your dataset directory, initialize it and create the dataloaders for training. 

Note: This time though, it is of great importance to make sure the dataset is not on your google drive but it is in your virtual machine storage. This can increase the training speed up to 400 times :)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 1. Imports from your repo
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.training.FlowModel_trainer import FlowTrainer, get_model_configs
from SCAPES.models.factorization import LocalEncoder
from SCAPES.models.flow import FlowModel

# Fill this with the path to your dataset
path_to_dataset = "path/to/your/dataset"  # Change this to your dataset path <---------------------------

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# dataset initialization (CPU is mandatory here, trainer moves batches to GPU)
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=[
        "latent_past", "latent_present", 
        "scale_past", "scale_present", 
        "clap_context_win", "index"
    ],
    device="cpu", # Leave dataset on CPU, trainer moves batches to GPU
    verbose=True
)

# create dataloaders for training and validation
train_split, val_split = dataset.get_splits()
train_loader = DataLoader(train_split, batch_size=16, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_split,   batch_size=16, shuffle=False)

## **2. Model Configuration**

Pick a model size between `small`, `medium` and `large`. If unsure just leave it as it is :)

In [ ]:
CONFIG_TYPE = "medium" # Change the size of yoru model <---------------------------

# Get the configurations to initialize the models based on the dataset and CONFIG_TYPE
LocalEncoder_config, FlowModel_config = get_model_configs(
    size             = CONFIG_TYPE,
    segment_length   = segment_length,
    frames_per_atom  = dataset.atoms_frames,         # Dynamically pulled
    atoms_hop_frames = dataset.atoms_hop_frames,    # Dynamically pulled
    crossfade_frames = dataset.crossfade_frames,    # Dynamically pulled
    LocalEncoder_time_entanglement = True,
    LocalEncoder_temporal_compression = 1
)

# Set your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the audio processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

# Initialize models using the dictionaries (**kwargs unpacking)
local_encoder = LocalEncoder(**LocalEncoder_config).to(device)
flow_model    = FlowModel(**FlowModel_config, device=device).to(device)

# count the parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"LocalEncoder parameters: {count_parameters(local_encoder):,}")
print(f"FlowModel parameters: {count_parameters(flow_model):,}")

# Initialize optimizer for both models
optimizer = AdamW(list(flow_model.parameters()) + list(local_encoder.parameters()), lr=1e-4)

## **3. Train!**

Although optional, you can listen to some audio reconstruction during training. In order to do so, fill the `TARGET_FILES` list with names of your files. Make sure to specify a model_path so you save your trained model.

In [ ]:
# PASS A LIST OF FILES YOU WANT TO MONITOR
TARGET_FILES = [] # Example files from your val split <---------------------------

# model path
MODEL_SAVE_PATH = "/content/model_name_here" # Make sure this is unique for your new model! <---------------------------

# Initialize the trainer
trainer = FlowTrainer(
    model           = flow_model,
    local_encoder   = local_encoder,
    train_loader    = train_loader,
    val_loader      = None,
    dataset         = dataset,
    processor       = processor_48k,
    optimizer       = optimizer,
    model_config    = FlowModel_config, 
    encoder_config  = LocalEncoder_config,  
    model_path      = MODEL_SAVE_PATH,
    context_source  = "clap",
    val_audio_files = TARGET_FILES,
    device          = device,
    past_dropout    = 0.25
)

# Start Training
trainer.train(epochs=50, audio_val_freq=10, val_nfe=16)

## **4. Save your model**

You will need your model for inference. Run the following cells to zip your model data and then **make sure you download it**. 

Note: You can technically just leave it in your Google Drive folder and load it from there to do inference, however that might be a bit slower.

In [ ]:
# Zip the model directory (recursively) using Linux zip
zip_path = f"{MODEL_SAVE_PATH.rstrip('/')}_data.zip"
!zip -r "{zip_path}" "{MODEL_SAVE_PATH}"

print(f"✅ Created: {zip_path}")

## **5. Restart the kernel (optional)**

It is maybe a good idea to restart the kernel to clean your variables :) If you do so, run the following cell so SCAPES can be loaded properly.

In [ ]:
# Move into repo root
%cd SCAPES

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ SCAPES setup complete. You can now do: from SCAPES import ...")

# **Task 3. Inference**

SCAPES model has many modes of inference. This notebook has examples for some of them.

## **1. Importing stuff**

In [ ]:
import torch
import json
from pathlib import Path
from IPython.display import Audio, display

from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.models.factorization import load_local_encoder, load_global_encoder
from SCAPES.models.flow import load_flow_model
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper
from SCAPES.inference.FlowInference import FlowInference, load_and_encode, run_interpolation_pipeline, run_resynthesis_pipeline

# Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# Initialize processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

## **2. Load your model and initialize the inference engine**

Point to the model checkpoint you want to use.

In [ ]:
# Flow model path
model_path = Path("path/to/your/model")  # Make sure this is the same as your training model path! <---------------------------

# Create paths for best models, models at epoch given and configs
best_flow_ckpt_path  = model_path / "checkpoints" / "best_flow_model.pt"
flow_config_path     = model_path / "checkpoints" / "flow_model_config.json"
best_local_ckpt_path = model_path / "checkpoints" / "best_local_encoder.pt"
local_config_path    = model_path / "checkpoints" / "local_encoder_config.json"

# Load checkpoints and configs
best_local_encoder = load_local_encoder(
    checkpoint_path=best_local_ckpt_path, 
    json_path=local_config_path, 
    device=device
)
best_flow_model = load_flow_model(
    checkpoint_path=best_flow_ckpt_path, 
    json_path=flow_config_path, 
    device=device
)
clap_model = CLAPWrapper(version="2023", use_cuda=True)

# Open the JSON to pull the exact geometry the model was trained on
with open(flow_config_path, 'r') as f:
    f_config = json.load(f)

# Load the inference engine that puts everything together. This is the main entry point for inference, and the one you will use for interpolation and resynthesis pipelines.
best_engine = FlowInference(
    model            = best_flow_model,
    local_encoder    = best_local_encoder,
    processor        = processor_48k,
    context_model    = clap_model,  # Passing clap or proxy
    segment_length   = segment_length,
    context_length   = context_length,
    atoms_frames     = f_config.get("frames_per_atom", 48),
    atoms_hop_frames = f_config.get("atoms_hop_frames", 15),
    crossfade_frames = f_config.get("crossfade_frames", 3),
    device           = device,
    verbose          = True
)

## **3. Inference examples**

## **3.1. Audio reconstruction**

Point to a particular audio file in your dataset and reconstruct it from its full latent representation.

In [ ]:
duration = 10 # You can reduce the duration of the audio generated if it takes too long to generate the audio

audio_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_tensor = best_engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000] # Shape [1, 2, T]
filename     = Path(audio_path).stem
print("Original:    ", filename)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

# Run resynthesis pipeline
output = run_resynthesis_pipeline(
    engine     = best_engine,
    audio_path = audio_path,
    duration   = duration, 
    play       = True,
    TF         = True,
    NFE        = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
)

## **3.2. Semantic reconstruction**

Point to a particular audio file in your dataset and reconstruct it from its semantics only.

In [ ]:
duration = 10 # You can reduce the duration of the audio generated if it takes too long to generate the audio

audio_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_tensor = best_engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000] # Shape [1, 2, T]
filename     = Path(audio_path).stem
print("Original:    ", filename)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine     = best_engine,
    audio_path = audio_path,
    duration   = duration,
    play       = True,
    TF         = "partial", # Start with a little help from the audio data. Drops the help after 0.5 seconds.
    NFE        = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
)

## **3.3. Semantic interpolation**

Point to two audio files and explore what is in the middle of them. You can choose between linear and spherical interpolation for the semantic embeddings interpolation.

In [ ]:
audio_start_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_start_tensor = best_engine.load_audio_to_tensor(audio_start_path)[:,:,:5*48000] # Shape [1, 2, T]
filename     = Path(audio_start_path).stem
print("Original_start:    ", filename)
display(Audio(audio_start_tensor.detach().cpu().numpy()[0], rate=48000))

audio_end_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_end_tensor = best_engine.load_audio_to_tensor(audio_end_path)[:,:,:5*48000] # Shape [1, 2, T]
filename     = Path(audio_end_path).stem
print("Original_end:     ", filename)
display(Audio(audio_end_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(
    engine         = best_engine,
    audio_path_1   = audio_start_path,
    audio_path_2   = audio_end_path,
    timeline_size  = 200, # Let's go for around 10 seconds
    stay_time      = 20,  # We will start with around 2 seconds of the first and end with around 2 seconds of the second, leaving around 8 seconds for the interpolation in the middle
    stickyness     = 3.0, # Adjust this to decide how sticky is the middle point between the two audios. 1.0 is linear, higher is stickier and lower is less sticky.
    plot_stickyness_curve = True,
    play           = True,
    NFE            = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
    context_static = False,  # If True, uses the first context embedding for each audio only
)